# Car Recognition
## Computer Vision Block — Model 1

**Dataset:** Stanford Car Dataset by Classes Folder (Kaggle – jutrera)

**Goal:** Classify a car photo into one of 196 make/model/year classes and extract `brand` and `model_year` to feed into the ML price prediction block.

**Integration:** `Car photo → CV (recognition) → brand + model_year → ML block (price prediction)`

**Class name format:** Each folder is named `{Make} {Model} {Year}`, e.g. `BMW 3 Series Sedan 2012`. Brand and year are parsed directly from the predicted class name.

## Project Setup
### Libraries and Settings

In [ ]:
!pip install -q transformers accelerate evaluate datasets pillow scikit-learn torchvision

In [ ]:
import io
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm import tqdm

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
import evaluate
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

print('GPU available:', torch.cuda.is_available())

## 1. Data Loading and Inspection

**Data source:** [Stanford Car Dataset by Classes Folder](https://www.kaggle.com/datasets/jutrera/stanford-car-dataset-by-classes-folder)

Attach to your Kaggle notebook via **'+ Add Input'**.

In [ ]:
# Explore folder structure first
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')
        for f in files[:2]:
            print('  ' * (level + 1) + f)

In [ ]:
# Load dataset — imagefolder assigns labels from subfolder names
# Adjust DATA_ROOT if the printed path above differs
DATA_ROOT = '/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder'

dataset = load_dataset('imagefolder', data_dir=DATA_ROOT)
dataset

In [ ]:
# Class names — each is "{Make} {Model} {Year}"
class_names = dataset['train'].features['label'].names
NUM_LABELS  = len(class_names)
label2id    = {name: i for i, name in enumerate(class_names)}
id2label    = {i: name for i, name in enumerate(class_names)}

print(f'Total classes: {NUM_LABELS}')
print('Example class names:', class_names[:5])

In [ ]:
# Helper: parse brand and model_year from a class name string
def parse_class(class_name):
    """
    Input:  'BMW 3 Series Sedan 2012'
    Output: brand='BMW', model_year=2012
    """
    brand = class_name.split()[0]
    year_match = re.search(r'(\d{4})$', class_name.strip())
    model_year = int(year_match.group(1)) if year_match else None
    return brand, model_year

# Test the parser
for name in class_names[:5]:
    brand, year = parse_class(name)
    print(f'  {name!r:40s} → brand={brand!r}, model_year={year}')

In [ ]:
# Use the pre-existing train/test split from the dataset if available
# Otherwise create one manually
if 'test' in dataset:
    split = dataset['train'].train_test_split(test_size=0.15, seed=42)
    our_dataset = DatasetDict({
        'train':      split['train'],
        'validation': split['test'],
        'test':       dataset['test']
    })
else:
    split = dataset['train'].train_test_split(test_size=0.3, seed=42)
    val_test = split['test'].train_test_split(test_size=0.5, seed=42)
    our_dataset = DatasetDict({
        'train':      split['train'],
        'validation': val_test['train'],
        'test':       val_test['test']
    })

our_dataset

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of unique brands
brands = [parse_class(name)[0] for name in class_names]
unique_brands, counts = np.unique(brands, return_counts=True)
sorted_idx = np.argsort(counts)[::-1]

plt.figure(figsize=(12, 4))
plt.bar(unique_brands[sorted_idx[:20]], counts[sorted_idx[:20]], color='steelblue')
plt.title('Top 20 Brands by Number of Model Classes')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of classes')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of model years across classes
years = [parse_class(name)[1] for name in class_names if parse_class(name)[1]]
plt.figure(figsize=(10, 4))
plt.hist(years, bins=20, color='steelblue', edgecolor='white')
plt.title('Distribution of Model Years in Dataset')
plt.xlabel('Year')
plt.ylabel('Number of classes')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images from training set
def show_samples(ds, rows, cols):
    samples = ds.shuffle(seed=42).select(np.arange(rows * cols))
    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for i in range(rows * cols):
        img = samples[i]['image']
        if not hasattr(img, 'convert'):
            img = Image.open(io.BytesIO(img['bytes']))
        label = id2label[samples[i]['label']]
        brand, year = parse_class(label)
        fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f'{brand} ({year})', fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(our_dataset['train'], rows=3, cols=5)

## 3. Preprocessing

Using `AutoImageProcessor` from `google/vit-base-patch16-224` — resizes to 224×224 and normalises pixel values, identical to the damage assessment notebook.

In [ ]:
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')

def transforms(batch):
    images = [img.convert('RGB') if hasattr(img, 'convert')
              else Image.open(io.BytesIO(img['bytes'])).convert('RGB')
              for img in batch['image']]
    inputs = processor(images, return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

from torchvision import transforms as T
augmentation = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomRotation(10),
])

def transforms_augmented(batch):
    images = [img.convert('RGB') if hasattr(img, 'convert')
              else Image.open(io.BytesIO(img['bytes'])).convert('RGB')
              for img in batch['image']]
    images = [augmentation(img) for img in images]
    inputs = processor(images, return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

---
## Iteration 1 — Zero-Shot Baseline with CLIP

**Objective:** Establish a zero-shot baseline for brand recognition using CLIP. Candidate labels are the unique car brands (not all 196 classes — that would be too many for CLIP).

**Model:** `openai/clip-vit-large-patch14` (same checkpoint as Week 7 exercise)

**Metric:** Brand-level accuracy (correct if predicted brand matches true brand).

In [ ]:
clip_detector = pipeline(
    model='openai/clip-vit-large-patch14',
    task='zero-shot-image-classification'
)

# Use brand names as candidate labels (simpler than 196 classes)
unique_brands_list = sorted(list(set(brands)))
candidate_labels_clip = [f'a photo of a {b} car' for b in unique_brands_list]
print(f'Number of candidate labels: {len(candidate_labels_clip)}')
print('Example:', candidate_labels_clip[:3])

In [ ]:
# Evaluate CLIP on brand recognition (test sample)
test_sample = our_dataset['test'].shuffle(seed=42).select(range(min(150, len(our_dataset['test']))))

true_brands_clip = []
pred_brands_clip = []

for sample in tqdm(test_sample):
    img = sample['image']
    if not hasattr(img, 'convert'):
        img = Image.open(io.BytesIO(img['bytes']))
    results = clip_detector(img, candidate_labels=candidate_labels_clip)
    pred_label = max(results, key=lambda x: x['score'])['label']
    pred_brand = pred_label.replace('a photo of a ', '').replace(' car', '')
    true_brand = parse_class(id2label[sample['label']])[0]
    true_brands_clip.append(true_brand)
    pred_brands_clip.append(pred_brand)

clip_brand_acc = accuracy_score(true_brands_clip, pred_brands_clip)
print(f'CLIP Zero-Shot Brand Accuracy: {clip_brand_acc:.4f}')

---
## Iteration 2 — Transfer Learning: ViT with Frozen Backbone

**Objective:** Fine-tune only the classification head of ViT on all 196 Stanford Cars classes.

**Model:** `google/vit-base-patch16-224` — same as Week 6 exercise.

**Key changes vs Iteration 1:** Task-specific supervised training; 196-class full classification (brand + model + year).

In [ ]:
processed_dataset = our_dataset.with_transform(transforms)

In [ ]:
vit_iter2 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Freeze all layers except the classifier head
for name, p in vit_iter2.named_parameters():
    if not name.startswith('classifier'):
        p.requires_grad = False

num_params       = sum(p.numel() for p in vit_iter2.parameters())
trainable_params = sum(p.numel() for p in vit_iter2.parameters() if p.requires_grad)
print(f'{num_params = :,} | {trainable_params = :,}')

In [ ]:
training_args_iter2 = TrainingArguments(
    output_dir='./vit-cars-iter2',
    per_device_train_batch_size=16,
    num_train_epochs=5,
    learning_rate=3e-4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    remove_unused_columns=False,
    logging_steps=50,
    report_to='none',
    disable_tqdm=True
)

trainer_iter2 = Trainer(
    model=vit_iter2,
    args=training_args_iter2,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed_dataset['train'],
    eval_dataset=processed_dataset['validation'],
    processing_class=processor
)

trainer_iter2.train()

In [ ]:
results_iter2 = trainer_iter2.evaluate(processed_dataset['test'])
print('Iteration 2 Test Results:', results_iter2)

preds_iter2  = trainer_iter2.predict(processed_dataset['test'])
y_pred_iter2 = preds_iter2.predictions.argmax(axis=1)
y_true       = preds_iter2.label_ids

# Brand-level accuracy (more meaningful than 196-class accuracy)
true_brands_iter2 = [parse_class(id2label[l])[0] for l in y_true]
pred_brands_iter2 = [parse_class(id2label[p])[0] for p in y_pred_iter2]
brand_acc_iter2   = accuracy_score(true_brands_iter2, pred_brands_iter2)

print(f'Class accuracy (196 classes): {results_iter2["eval_accuracy"]:.4f}')
print(f'Brand accuracy:               {brand_acc_iter2:.4f}')

---
## Iteration 3 — Transfer Learning: ViT with Partial Unfreeze + Augmentation

**Objective:** Unfreeze the last 2 transformer encoder blocks and add augmentation to improve generalisation across car models.

**Key changes vs Iteration 2:** Encoder layers 10 and 11 unfrozen; random flip, colour jitter, and rotation augmentation added.

In [ ]:
processed_train_aug = our_dataset['train'].with_transform(transforms_augmented)
processed_val_test  = our_dataset.with_transform(transforms)

In [ ]:
vit_iter3 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

for name, p in vit_iter3.named_parameters():
    p.requires_grad = False

for name, p in vit_iter3.named_parameters():
    if ('encoder.layer.10' in name or
        'encoder.layer.11' in name or
        name.startswith('classifier')):
        p.requires_grad = True

num_params       = sum(p.numel() for p in vit_iter3.parameters())
trainable_params = sum(p.numel() for p in vit_iter3.parameters() if p.requires_grad)
print(f'{num_params = :,} | {trainable_params = :,}')

In [ ]:
training_args_iter3 = TrainingArguments(
    output_dir='./vit-cars-iter3',
    per_device_train_batch_size=16,
    num_train_epochs=8,
    learning_rate=1e-4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    remove_unused_columns=False,
    logging_steps=50,
    report_to='none',
    disable_tqdm=True
)

trainer_iter3 = Trainer(
    model=vit_iter3,
    args=training_args_iter3,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed_train_aug,
    eval_dataset=processed_val_test['validation'],
    processing_class=processor
)

trainer_iter3.train()

In [ ]:
results_iter3 = trainer_iter3.evaluate(processed_val_test['test'])
print('Iteration 3 Test Results:', results_iter3)

preds_iter3  = trainer_iter3.predict(processed_val_test['test'])
y_pred_iter3 = preds_iter3.predictions.argmax(axis=1)

true_brands_iter3 = [parse_class(id2label[l])[0] for l in y_true]
pred_brands_iter3 = [parse_class(id2label[p])[0] for p in y_pred_iter3]
brand_acc_iter3   = accuracy_score(true_brands_iter3, pred_brands_iter3)

print(f'Class accuracy (196 classes): {results_iter3["eval_accuracy"]:.4f}')
print(f'Brand accuracy:               {brand_acc_iter3:.4f}')

In [ ]:
# --- Model Comparison Summary ---
print('=== Model Comparison ===')
print(f'Iter 1 – CLIP zero-shot brand accuracy:        {clip_brand_acc:.4f}')
print(f'Iter 2 – ViT frozen backbone class accuracy:   {results_iter2["eval_accuracy"]:.4f}  |  brand: {brand_acc_iter2:.4f}')
print(f'Iter 3 – ViT partial unfreeze + aug accuracy:  {results_iter3["eval_accuracy"]:.4f}  |  brand: {brand_acc_iter3:.4f}')

## 4. Error Analysis

In [ ]:
# Visual inspection of predictions — best model (Iter 3)
def show_predictions(rows, cols):
    samples = our_dataset['test'].shuffle(seed=99).select(np.arange(rows * cols))
    processed_samples = samples.with_transform(transforms)
    predictions = trainer_iter3.predict(processed_samples).predictions.argmax(axis=1)
    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for i in range(rows * cols):
        img = samples[i]['image']
        if not hasattr(img, 'convert'):
            img = Image.open(io.BytesIO(img['bytes']))
        true_class = id2label[samples[i]['label']]
        pred_class = id2label[predictions[i]]
        true_brand, true_year = parse_class(true_class)
        pred_brand, pred_year = parse_class(pred_class)
        colour = 'green' if true_brand == pred_brand else 'red'
        fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(
            f'True: {true_brand} {true_year}\nPred: {pred_brand} {pred_year}',
            color=colour, fontsize=8
        )
        plt.axis('off')
    plt.suptitle('Sample Predictions (green=correct brand, red=wrong)', fontsize=12)
    plt.tight_layout()
    plt.show()

show_predictions(rows=3, cols=5)

## 5. Save Final Model

In [ ]:
vit_iter3.save_pretrained('./car_recognition_model')
processor.save_pretrained('./car_recognition_model')
print('Model saved to ./car_recognition_model/')

## 6. Integration Test — Extract brand + model_year from a Single Image

Demonstrates how the ML block calls this function at inference time.

In [ ]:
car_recognizer = pipeline(
    task='image-classification',
    model=vit_iter3,
    image_processor=processor
)

def predict_car_identity(image):
    """
    Returns brand (str) and model_year (int) from a car photo.
    These feed directly into the ML price prediction block.
    """
    if isinstance(image, str):
        image = Image.open(image).convert('RGB')
    result = car_recognizer(image)
    top = max(result, key=lambda x: x['score'])
    brand, model_year = parse_class(top['label'])
    print(f'Predicted class: {top["label"]}  (confidence: {top["score"]:.2%})')
    print(f'→ brand: {brand!r},  model_year: {model_year}')
    return brand, model_year

# Test on one image from the test set
sample = our_dataset['test'][0]
img = sample['image']
if not hasattr(img, 'convert'):
    img = Image.open(io.BytesIO(img['bytes']))

brand, year = predict_car_identity(img)
print(f'True class: {id2label[sample["label"]]}')